In [1]:
import gzip
import numpy as np
import pandas as pd


In [2]:
# HLA-DQA1 is not listed because it consistently didn't pass QC in all datasets
classical_class_I_II = {'HLA-A', 'HLA-B', 'HLA-C', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DQA1', 'HLA-DQB1', 'HLA-DRA', 'HLA-DRB1'}

# MACs
mac_thresholds = [5, 10, 20, 50]

In [3]:
nocov_tsv = {
    '2field': '../data/1000G_ONT/GWAS_continuous/2field_alleles/seed_421_nocov.tsv.gz',
    '4field': '../data/1000G_ONT/GWAS_continuous/4field_alleles/seed_421_nocov.tsv.gz',
}

cov_tsv = {
    '2field': '../data/1000G_ONT/GWAS_continuous/2field_alleles/seed_421_cov.tsv.gz',
    '4field': '../data/1000G_ONT/GWAS_continuous/4field_alleles/seed_421_cov.tsv.gz',
}

In [4]:
def read_pvalue_tsv(filepath, mac):
    df = pd.read_csv(filepath, sep = "\t")
    df['gene'] = df['allele'].apply(lambda x: x.split('*')[0])

    assert len(df[~df.error.isna()]) == 0 # Any convergence errors? 
    
    df = df[df.gene.isin(classical_class_I_II)] # Keep only classical class I and II
        
    df_temp = df[df.mac >= mac]

    # Get all unique alleles and make sure that they all had the same AC values
    df_alleles = df_temp.groupby('allele')[['ac']].nunique().reset_index()
    assert len(df_alleles[df_alleles.ac != 1]) == 0 
    n_alleles = len(df_alleles)

    # Get single minimal p-value per simulation
    df_pvalues = df_temp[["output", "pheno", "p"]].groupby(["output", "pheno"]).min().reset_index()
    n_simulations = len(df_pvalues)
    min_pvalues_q5 = np.quantile(df_pvalues.p, 0.05)
    m_eff = 0.05 / min_pvalues_q5
        
    return n_simulations, n_alleles, min_pvalues_q5, m_eff



In [5]:
print("With covariates")

print(f'Dataset;', end = "")
for mac in mac_thresholds:
    print(f';N_alleles [MAC>={mac}]; M_eff [MAC>={mac}]', end = "")
print()

for label in ['2field', '4field']:
    print(f'{label}', end = "")
    for mac in mac_thresholds:
        n_simulations, n_alleles, min_pvalues_q5, m_eff = read_pvalue_tsv(cov_tsv[label], mac)
        print(f';{n_alleles};{m_eff:.0f} ({m_eff / n_alleles * 100:.0f})', end = "")
    print()
    

With covariates
Dataset;;N_alleles [MAC>=5]; M_eff [MAC>=5];N_alleles [MAC>=10]; M_eff [MAC>=10];N_alleles [MAC>=20]; M_eff [MAC>=20];N_alleles [MAC>=50]; M_eff [MAC>=50]
2field;175;164 (94);137;124 (91);94;86 (91);46;41 (89)
4field;273;243 (89);183;167 (91);109;101 (92);28;25 (89)


In [6]:
print("Without covariates")

print(f'Dataset;', end = "")
for mac in mac_thresholds:
    print(f';N_alleles [MAC>={mac}]; M_eff [MAC>={mac}]', end = "")
print()

for label in ['2field', '4field']:
    print(f'{label}', end = "")
    for mac in mac_thresholds:
        n_simulations, n_alleles, min_pvalues_q5, m_eff = read_pvalue_tsv(nocov_tsv[label], mac)
        print(f';{n_alleles};{m_eff:.0f} ({m_eff / n_alleles * 100:.0f})', end = "")
    print()

Without covariates
Dataset;;N_alleles [MAC>=5]; M_eff [MAC>=5];N_alleles [MAC>=10]; M_eff [MAC>=10];N_alleles [MAC>=20]; M_eff [MAC>=20];N_alleles [MAC>=50]; M_eff [MAC>=50]
2field;175;159 (91);137;123 (89);94;86 (92);46;41 (89)
4field;273;243 (89);183;162 (89);109;98 (90);28;25 (89)
